# Article 7: Security Guardrails Benchmark Analysis

Benchmarks the regex-only layer of `GuardrailsManager` against 104 red-team prompts across three severity tiers (L1 naive, L2 moderate, L3 advanced).  
Key metrics: block rate by severity tier, false positive rate, and `check_input` latency percentiles.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [ ]:
data = json.loads(Path("../results/data/article_07_benchmarks.json").read_text())
data

In [ ]:
os.makedirs("../results/charts/article_07", exist_ok=True)

severity_levels = ["L1", "L2", "L3"]
block_rates = [
    data["block_rate_by_severity"][lvl] * 100 for lvl in severity_levels
]

# Color encodes whether the tier meets the 90% target from specs (FR-5.3.1).
# Red = below target, green = at or above target.
TARGET_PCT = 90.0
colors = ["#43a047" if r >= TARGET_PCT else "#e53935" for r in block_rates]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(severity_levels, block_rates, color=colors, width=0.5)

# 90% target line - from Article 7 spec (FR-5.3.1: block >90% of L2+ attacks).
ax.axhline(TARGET_PCT, color="navy", linestyle="--", linewidth=1.5, label="90% target")

ax.set_xlabel("Attack Severity Tier")
ax.set_ylabel("Block Rate (%)")
ax.set_title("Guardrail Block Rate by Severity - Regex Layer Only")
ax.set_ylim(0, 115)
ax.legend()

for bar, rate in zip(bars, block_rates):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        rate + 2,
        f"{rate:.1f}%",
        ha="center",
        fontweight="bold",
    )

plt.tight_layout()
plt.savefig(
    "../results/charts/article_07/01_block_rate_by_severity.png", dpi=150
)
plt.show()
print(f"Saved: 01_block_rate_by_severity.png")

In [ ]:
categories = sorted(data["block_rate_by_category"].keys())
cat_rates = [data["block_rate_by_category"][c] * 100 for c in categories]

# Horizontal bars make long category names readable without rotation.
colors_cat = ["#43a047" if r >= TARGET_PCT else "#e53935" for r in cat_rates]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(categories, cat_rates, color=colors_cat)
ax.axvline(TARGET_PCT, color="navy", linestyle="--", linewidth=1.5, label="90% target")

ax.set_xlabel("Block Rate (%) - attack rows only")
ax.set_title("Guardrail Block Rate by Attack Category - Regex Layer Only")
ax.set_xlim(0, 120)
ax.legend()

for bar, rate in zip(bars, cat_rates):
    ax.text(
        rate + 1,
        bar.get_y() + bar.get_height() / 2,
        f"{rate:.1f}%",
        va="center",
        fontweight="bold",
    )

plt.tight_layout()
plt.savefig(
    "../results/charts/article_07/02_block_rate_by_category.png", dpi=150
)
plt.show()
print(f"Saved: 02_block_rate_by_category.png")

In [ ]:
print("Latency Metrics (check_input over all 104 prompts)")
print(f"  p50:  {data['latency_p50_ms']:.3f} ms")
print(f"  p95:  {data['latency_p95_ms']:.3f} ms")
print(f"  p99:  {data['latency_p99_ms']:.3f} ms")
print()
print("False Positive Rate")
fp_pct = data['false_positive_rate'] * 100
fp_status = "OK (below 5% threshold)" if fp_pct < 5 else "ABOVE 5% THRESHOLD"
print(f"  {fp_pct:.1f}%  ->  {fp_status}")
print()
print("Coverage Summary")
print(f"  {data['prompts_blocked']}/{data['prompts_expected_blocked']} attack prompts blocked")
print(f"  Total prompts evaluated: {data['total_prompts']}")

## Analysis

The regex-only guardrail layer catches **100% of PII-containing inputs** (SSN, phone, email patterns) and has **zero false positives** on the five benign test queries - but its overall attack block rate falls well below the 90% target across all severity tiers.  
L1 naive attacks are blocked at only 33%, because most L1 prompts use phrasing the current four jailbreak regexes do not match verbatim (e.g., "Forget everything above", "In this roleplay...").  
This gap motivates the semantic fallback layer (Llama-Guard / NeMo), which can catch paraphrased and role-play injection patterns that no static regex can enumerate; the latency cost of that layer (200-500ms) is acceptable because it only runs after the fast regex pass (<0.01ms p99).